Load the three datasets

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table(
    "customer360_dev.bronze.customers"
)

silver_df = spark.table(
    "customer360_dev.silver.customers"
)

reject_df = spark.table(
    "customer360_dev.quarantine.customer_rejects"
)

Calculate Bronze, Silver, and rejected counts

In [0]:
bronze_count = bronze_df.count()
silver_count = silver_df.count()
reject_count = reject_df.count()

print(f"Bronze rows     : {bronze_count:,}")
print(f"Silver rows     : {silver_count:,}")
print(f"Rejected rows   : {reject_count:,}")

Bronze rows     : 1,000,000
Silver rows     : 986,000
Rejected rows   : 9,000


Recreate only the validation required to identify valid candidates

In [0]:
validated_bronze_df = (
    bronze_df
    .withColumn(
        "date_of_birth_cast",
        F.to_date("date_of_birth", "yyyy-MM-dd")
    )
    .withColumn(
        "email_clean",
        F.lower(F.trim("email"))
    )
    .withColumn(
        "customer_status_clean",
        F.upper(F.trim("customer_status"))
    )
    .withColumn(
        "dq_missing_customer_id",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )
    .withColumn(
        "dq_invalid_email",
        ~F.col("email_clean").rlike(
            r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
        )
    )
    .withColumn(
        "dq_invalid_status",
        ~F.col("customer_status_clean").isin(
            "ACTIVE",
            "INACTIVE",
            "SUSPENDED"
        )
    )
    .withColumn(
        "dq_invalid_dob",
        F.col("date_of_birth_cast").isNull()
        | (F.col("date_of_birth_cast") > F.current_date())
    )
    .withColumn(
        "dq_critical_failure",
        F.col("dq_missing_customer_id")
        | F.col("dq_invalid_email")
        | F.col("dq_invalid_status")
        | F.col("dq_invalid_dob")
    )
)

Count valid candidates before deduplication

In [0]:
valid_candidate_df = (
    validated_bronze_df
    .filter(~F.col("dq_critical_failure"))
)

valid_candidate_count = valid_candidate_df.count()

print(
    f"Valid candidates before dedup: "
    f"{valid_candidate_count:,}"
)

Valid candidates before dedup: 991,000


Calculate how many valid duplicates were removed

In [0]:
duplicate_rows_removed = (
    valid_candidate_count - silver_count
)

print(
    f"Valid duplicate rows removed: "
    f"{duplicate_rows_removed:,}"
)

Valid duplicate rows removed: 5,000


Perform the reconciliation equation

In [0]:
accounted_for = (
    silver_count
    + reject_count
    + duplicate_rows_removed
)

difference = bronze_count - accounted_for

print(f"Bronze rows       : {bronze_count:,}")
print(f"Silver rows       : {silver_count:,}")
print(f"Rejected rows     : {reject_count:,}")
print(f"Duplicates removed: {duplicate_rows_removed:,}")
print(f"Accounted for     : {accounted_for:,}")
print(f"Difference        : {difference:,}")

Bronze rows       : 1,000,000
Silver rows       : 986,000
Rejected rows     : 9,000
Duplicates removed: 5,000
Accounted for     : 1,000,000
Difference        : 0


Create a clear reconciliation result

In [0]:
reconciliation_result = [
    {
        "entity": "customers",
        "bronze_rows": bronze_count,
        "silver_rows": silver_count,
        "quarantine_rows": reject_count,
        "duplicates_removed": duplicate_rows_removed,
        "accounted_for_rows": accounted_for,
        "difference": difference,
        "status": "PASS" if difference == 0 else "FAIL"
    }
]

display(
    spark.createDataFrame(reconciliation_result)
)

accounted_for_rows,bronze_rows,difference,duplicates_removed,entity,quarantine_rows,silver_rows,status
1000000,1000000,0,5000,customers,9000,986000,PASS


Validate the reject reasons

In [0]:
(
    reject_df
    .groupBy("rejection_reason")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+-----------------------+-----+
|rejection_reason       |count|
+-----------------------+-----+
|INVALID_EMAIL          |5000 |
|INVALID_CUSTOMER_STATUS|2000 |
|MISSING_CUSTOMER_ID    |1000 |
|INVALID_DATE_OF_BIRTH  |1000 |
+-----------------------+-----+



Check for duplicate customer IDs in Silver

In [0]:
silver_duplicate_keys = (
    silver_df
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(
    "Duplicate customer IDs in Silver:",
    silver_duplicate_keys
)

Duplicate customer IDs in Silver: 0


Check Silver business-key nulls

In [0]:
silver_missing_keys = (
    silver_df
    .filter(
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )
    .count()
)

print(
    "Missing customer IDs in Silver:",
    silver_missing_keys
)

Missing customer IDs in Silver: 0
